# Budget Auction — Experiment 2: Multi-Good Privately Correlated Budget Auction

Compares mechanism input variants **A** (GBAM/public-balance), **B** (richer history), **C** (full history) in a setting *beyond* Deng 2021: types are multi-dimensional and transitions depend on private type history (private correlation).

The key hypothesis: richer history compression should outperform the GBAM-style compressed state here, because the simple public balance is no longer theoretically sufficient.

In [ ]:
%pip install -q git+https://github.com/deepmind/dm-haiku
%pip install -q jaxopt optax flax pyyaml


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/My\ Research/Dynamic\ Mechanism\ Design/Budget\ Auction
except ImportError:
    pass


In [ ]:
from typing import Dict
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
import pickle, json, os

from budget_auction.core import (
    BudgetSimulator, BoundedContPolicy,
    build_bounded_policy_net, plot_loss_dict,
)
from budget_auction.mechanisms.budget_mechanism import BudgetMechanism
from budget_auction.mechanisms.input_adapters import VARIANT_NETWORK
from budget_auction.envs.private_budget_multigood_env import create_private_budget_multigood_auction
from budget_auction.baselines.static_myopic import run_static_myopic_baseline
from budget_auction.train.run_acceptance import nested_GD_budget
from budget_auction.train.evaluate import eval_budget_mechanism_multikey, print_eval_report

print('JAX devices:', jax.devices())


## Configuration

In [ ]:
VARIANT = 'A'   # 'A' | 'B' | 'C'
SEED    = 42

cfg = dict(
    n_buyers=2, n_goods=2, horizon=5, discount=0.9,
    initial_budget=3.0,
    profit_goal=2.0,
    num_outer_epochs=2000,
    num_inner_steps=300,
    lr_theta=1e-4, lr_w=3e-4,
    num_samples_train=64,
    alpha_weight=1e5, beta_weight=1e3,
    print_iter=100,
    policy_hidden=128, residual_init_std=1.0,
)
print('Config:', cfg)


## Build Auction Environment

In [ ]:
auction = create_private_budget_multigood_auction(
    n_buyers=cfg['n_buyers'], n_goods=cfg['n_goods'],
    horizon=cfg['horizon'],  discount=cfg['discount'],
    initial_budget=cfg['initial_budget'], seed=SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}')
print(f'Initial budgets: {auction.initial_budgets}')
print(f'Type space: [{auction.types_low}, {auction.types_high}]')


In [ ]:
key = jax.random.PRNGKey(0)
s0 = auction.get_initial_state(key)
print('Initial state:')
print(f'  types  = {s0.current_type}   (shape: {s0.current_type.shape})')
print(f'  alpha  = {s0.alpha}   (should be 0)')
print(f'  beta   = {s0.beta}    (should be {auction.initial_budgets})')


## Check Type Transition Dynamics

In [ ]:
# Verify private correlation: types at t+1 should be near types at t
from budget_auction.envs.private_budget_multigood_env import private_budget_transition_fn

t0_types = s0.current_type  # (B, G)
alloc_hist = jnp.zeros((auction.num_horizons, auction.num_buyers, auction.num_goods))
keys_trans = jax.random.split(jax.random.PRNGKey(1), 20)
samples = [auction.transition_function(auction, 0, t0_types, alloc_hist, k) for k in keys_trans]
samples = jnp.stack(samples)  # (20, B, G)
print(f'Current types: {t0_types}')
print(f'Next types mean: {jnp.mean(samples, 0)}')
print(f'Next types std:  {jnp.std(samples, 0)}')
print('(Should be close to current types with small sigma_no=0.01 and no allocation)')


## Build Mechanism Network (Quick Single-Variant Check)

In [ ]:
raw_fn = VARIANT_NETWORK[VARIANT]
mechanism_network = hk.without_apply_rng(hk.transform(raw_fn))
policy_network    = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub, G=auction.num_goods,
    hidden=cfg['policy_hidden'], residual_init_std=cfg['residual_init_std'],
)
print(f'Mechanism variant: {VARIANT}  (n_goods={auction.num_goods})')


## Evaluate Myopic Baseline

In [ ]:
print('--- Baseline B1: Static Myopic ---')
b1 = run_static_myopic_baseline(auction, num_samples=128, num_keys=8, base_seed=1)
for k, v in b1.items():
    if isinstance(v, float): print(f'  {k}: {v:.4f}')


## Quick Single-Variant Training

Run a single variant with default hyperparameters as a sanity check. Skip to the **Hyperparameter Search** section below for the full pipeline.

In [ ]:
logs = nested_GD_budget(
    auction=auction,
    mechanism_network=mechanism_network,
    policy_network=policy_network,
    profit_goal=cfg['profit_goal'],
    num_outer_epochs=cfg['num_outer_epochs'],
    lr_theta=cfg['lr_theta'],
    num_inner_steps=cfg['num_inner_steps'],
    lr_w=cfg['lr_w'],
    num_samples_train=cfg['num_samples_train'],
    alpha_weight=cfg['alpha_weight'],
    beta_weight=cfg['beta_weight'],
    print_iter=cfg['print_iter'],
    seed=SEED,
)
mechanism_params = logs.pop('mechanism_params')
policy_params    = logs.pop('policy_params')
print('Single-variant training complete.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(logs['reg_plus']); axes[0].set_title('Regret+ ' + VARIANT); axes[0].set_xlabel('Epoch')
axes[1].plot(logs['gap_plus']); axes[1].set_title('Profit Gap+');        axes[1].set_xlabel('Epoch')
axes[2].plot(logs['ir']);        axes[2].set_title('IR Loss');            axes[2].set_xlabel('Epoch')
plt.suptitle(f'Exp 2 — Single-variant quick check: Variant {VARIANT} ({auction.num_goods} goods)', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
eval_keys = jax.random.split(jax.random.PRNGKey(9999), 32)
results = eval_budget_mechanism_multikey(
    auction=auction,
    mechanism_network=mechanism_network,
    mechanism_params=mechanism_params,
    profit_goal=cfg['profit_goal'],
    num_samples=256, keys=eval_keys,
    policy_network=policy_network,
    policy_params=policy_params,
)
print_eval_report(results, prefix=f'Single-variant {VARIANT}: ')


## Hyperparameter Search

Systematically explores hyperparameter combinations within a configurable time budget (default **12 hours**). Results are saved to disk after every trial so the search can be resumed if interrupted. After the search, the best configuration is used to retrain for the full epoch budget, and all three variants A / B / C are compared under the same optimal hyperparameters.

**Search space:** `lr_theta`, `lr_w`, `num_inner_steps`, `alpha_weight`, `beta_weight`  
**Scoring:** maximize revenue subject to low IR- and budget-violation rates.

In [ ]:
import itertools, random, time

# ---- HP Search Configuration (edit these if needed) ----
HP_TIME_BUDGET_HOURS = 12        # max total wall-clock search time
HP_TRIAL_EPOCHS      = 500       # outer epochs per screening trial
HP_TRIAL_INNER       = 150       # inner gradient steps per trial
HP_TRIAL_SAMPLES     = 64        # MC samples per trial
HP_EVAL_KEYS         = 16        # number of eval keys per trial
HP_EVAL_SAMPLES      = 128       # MC samples per trial evaluation
HP_IR_VIOL_THRESH    = 0.05      # acceptable IR violation rate
HP_BV_THRESH         = 0.05      # acceptable budget violation rate

# ---- Search space (grid, randomly ordered = random search) ----
hp_search_space = {
    'lr_theta':        [3e-5, 1e-4, 3e-4],
    'lr_w':            [1e-4, 3e-4, 1e-3],
    'num_inner_steps': [150, 300],
    'alpha_weight':    [1e4, 1e5, 1e6],
    'beta_weight':     [1e3, 1e4],
}

_hp_keys   = list(hp_search_space.keys())
_hp_combos = list(itertools.product(*[hp_search_space[k] for k in _hp_keys]))
random.seed(SEED)
random.shuffle(_hp_combos)
print(f'Search space: {len(_hp_combos)} combinations | budget: {HP_TIME_BUDGET_HOURS}h')
print(f'Trial: {HP_TRIAL_EPOCHS} epochs x {HP_TRIAL_INNER} inner steps')

hp_save_dir = 'outputs/private_budget_multigood/hparam_search'
os.makedirs(hp_save_dir, exist_ok=True)

def _hp_score(r):
    """Score a trial: revenue minus feasibility-violation penalty."""
    bv_pen = 10.0 * max(0.0, r['budget_viol_mean'] - HP_BV_THRESH)
    iv_pen = 10.0 * max(0.0, r['ir_viol_mean']    - HP_IR_VIOL_THRESH)
    return r['revenue_mean'] - 0.3 * r['ir_loss_mean'] - bv_pen - iv_pen


In [ ]:
# ---- Load existing results if resuming an interrupted search ----
hp_results_path = os.path.join(hp_save_dir, 'trial_results.json')
if os.path.exists(hp_results_path):
    with open(hp_results_path) as _f:
        all_trial_results = json.load(_f)
    print(f'Resuming: {len(all_trial_results)} completed trials loaded')
    _done_keys = {str(r['config']) for r in all_trial_results}
else:
    all_trial_results = []
    _done_keys = set()

_search_start = time.time()
_budget_s     = HP_TIME_BUDGET_HOURS * 3600

for _tidx, _combo in enumerate(_hp_combos):
    _elapsed = time.time() - _search_start
    if _elapsed >= _budget_s:
        print(f'Time budget ({HP_TIME_BUDGET_HOURS}h) reached after {_tidx} combos.')
        break

    _tcfg = dict(zip(_hp_keys, _combo))
    if str(_tcfg) in _done_keys:
        continue  # skip already-completed (enables resumption)

    _rem_h = (_budget_s - _elapsed) / 3600
    print(f'Trial {_tidx+1}/{len(_hp_combos)} | elapsed={_elapsed/3600:.2f}h rem={_rem_h:.2f}h')
    print(f'  cfg: {_tcfg}')

    _trial_seed = SEED + 10000 + _tidx
    try:
        _mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
        _pnet = build_bounded_policy_net(
            d_priv=auction.d_priv, d_pub=auction.d_pub,
            G=auction.num_goods, hidden=cfg['policy_hidden'],
            residual_init_std=cfg['residual_init_std'],
        )
        _tlogs = nested_GD_budget(
            auction=auction,
            mechanism_network=_mnet, policy_network=_pnet,
            profit_goal=cfg['profit_goal'],
            num_outer_epochs=HP_TRIAL_EPOCHS,
            lr_theta=_tcfg['lr_theta'],
            num_inner_steps=_tcfg['num_inner_steps'],
            lr_w=_tcfg['lr_w'],
            num_samples_train=HP_TRIAL_SAMPLES,
            alpha_weight=_tcfg['alpha_weight'],
            beta_weight=_tcfg['beta_weight'],
            print_iter=10**7,  # suppress per-epoch output during search
            seed=_trial_seed,
        )
        _tmp = _tlogs.pop('mechanism_params')
        _tpp = _tlogs.pop('policy_params')

        _ekeys = jax.random.split(jax.random.PRNGKey(_trial_seed + 99999), HP_EVAL_KEYS)
        _tres = eval_budget_mechanism_multikey(
            auction=auction, mechanism_network=_mnet, mechanism_params=_tmp,
            profit_goal=cfg['profit_goal'],
            num_samples=HP_EVAL_SAMPLES, keys=_ekeys,
            policy_network=_pnet, policy_params=_tpp,
        )

        _rec = {
            'trial_idx':        _tidx,
            'config':           _tcfg,
            'revenue_mean':     float(_tres['profit_stats']['mean']),
            'revenue_std':      float(_tres['profit_stats']['std']),
            'ir_loss_mean':     float(_tres['ir_loss_stats']['mean']),
            'budget_viol_mean': float(_tres['budget_viol_stats']['mean']),
            'ir_viol_mean':     float(_tres['ir_viol_stats']['mean']),
            'gap_plus_mean':    float(_tres['gap_plus_stats']['mean']),
            'regret_mean':      float(_tres.get('regret_stats', {}).get('mean', float('nan'))),
            'final_reg_train':  float(_tlogs['reg'][-1]) if _tlogs.get('reg') else float('nan'),
            'final_ir_train':   float(_tlogs['ir'][-1])  if _tlogs.get('ir')  else float('nan'),
            'elapsed_h':        _elapsed / 3600,
            'trial_seed':       _trial_seed,
        }
        _rec['score'] = _hp_score(_rec)
        all_trial_results.append(_rec)
        _done_keys.add(str(_tcfg))

        # Fault-tolerant: persist after every trial
        with open(hp_results_path, 'w') as _f:
            json.dump(all_trial_results, _f, indent=2)

        _rev = _rec['revenue_mean']; _ir = _rec['ir_loss_mean']
        _bv  = _rec['budget_viol_mean']; _sc = _rec['score']
        print(f'  => revenue={_rev:.4f}  ir_loss={_ir:.4f}  bv={_bv:.4f}  score={_sc:.4f}')

    except Exception as _exc:
        print(f'  [TRIAL FAILED] {_exc}')

_total_time_h = (time.time() - _search_start) / 3600
print(f'Search complete: {len(all_trial_results)} successful trials ({_total_time_h:.2f}h)')


In [ ]:
# ---- Summarise search results ----
_sorted_trials = sorted(all_trial_results, key=lambda r: r['score'], reverse=True)

print(f'Top 10 / {len(all_trial_results)} trials by score:')
print(f'  {"score":>7}  {"revenue":>8}  {"ir_loss":>8}  {"bv_rate":>7}  {"iv_rate":>7}  config')
print('  ' + '-'*95)
for _r in _sorted_trials[:10]:
    _sc  = _r['score'];             _rev = _r['revenue_mean']
    _ir  = _r['ir_loss_mean'];      _bv  = _r['budget_viol_mean']
    _iv  = _r['ir_viol_mean'];      _cfg = _r['config']
    print(f'  {_sc:>7.4f}  {_rev:>8.4f}  {_ir:>8.4f}  {_bv:>7.4f}  {_iv:>7.4f}  {_cfg}')

# ---- Visualise ----
_revenues  = [r['revenue_mean']           for r in all_trial_results]
_ir_losses = [r['ir_loss_mean']            for r in all_trial_results]
_scores    = [r['score']                   for r in all_trial_results]
_lr_thetas = [r['config']['lr_theta']       for r in all_trial_results]
_alpha_ws  = [r['config']['alpha_weight']   for r in all_trial_results]
_beta_ws   = [r['config']['beta_weight']    for r in all_trial_results]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

_sc_plot = axes[0].scatter(_revenues, _ir_losses, c=_scores,
                           cmap='viridis', alpha=0.75, s=40)
axes[0].set_xlabel('Revenue'); axes[0].set_ylabel('IR Loss')
axes[0].set_title('Exp 2 — Revenue vs IR Loss (color=score)')
plt.colorbar(_sc_plot, ax=axes[0])

axes[1].hist(_scores, bins=20, edgecolor='black', alpha=0.75, color='steelblue')
_best_sc = _sorted_trials[0]['score']
axes[1].axvline(_best_sc, color='red', linestyle='--', label=f'Best={_best_sc:.3f}')
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution'); axes[1].legend()

for _lr in sorted(set(_lr_thetas)):
    _mask = [r for r in all_trial_results if r['config']['lr_theta'] == _lr]
    axes[2].scatter([_lr]*len(_mask), [r['score'] for r in _mask],
                    alpha=0.6, s=30, label=f'{_lr:.0e}')
axes[2].set_xscale('log'); axes[2].set_xlabel('lr_theta')
axes[2].set_ylabel('Score')
axes[2].set_title('Exp 2 — Score vs lr_theta')
axes[2].legend(title='lr_theta', fontsize=8)

plt.tight_layout(); plt.show()


In [ ]:
# ---- Select best configuration ----
best_trial  = max(all_trial_results, key=lambda r: r['score'])
best_cfg_hp = best_trial['config']

print('=' * 62)
print('OPTIMAL HYPERPARAMETERS — Exp 2')
print('=' * 62)
for _k, _v in best_cfg_hp.items():
    print(f'  {_k:20s}: {_v}')
print()
print(f'  Screening revenue : {best_trial["revenue_mean"]:.4f} +/- {best_trial["revenue_std"]:.4f}')
print(f'  Screening IR loss : {best_trial["ir_loss_mean"]:.4f}')
print(f'  Budget viol rate  : {best_trial["budget_viol_mean"]:.4f}')
print(f'  IR viol rate      : {best_trial["ir_viol_mean"]:.4f}')
print(f'  Score             : {best_trial["score"]:.4f}')

_best_save = {
    'optimal_config':  best_cfg_hp,
    'screening_stats': {k: v for k, v in best_trial.items() if k != 'config'},
    'variant':         VARIANT,
    'search_space':    hp_search_space,
    'num_trials':      len(all_trial_results),
    'ranked_configs':  [
        {'config': r['config'], 'score': r['score'],
         'revenue': r['revenue_mean'], 'ir_loss': r['ir_loss_mean']}
        for r in _sorted_trials
    ],
}
with open(os.path.join(hp_save_dir, 'best_hparams.json'), 'w') as _f:
    json.dump(_best_save, _f, indent=2)
print(f'\nSaved: {hp_save_dir}/best_hparams.json')


## Final Training with Optimal Hyperparameters

Retrain for the full epoch budget using the best configuration found above.

In [ ]:
print(f'Retraining with optimal config for {cfg["num_outer_epochs"]} epochs ...')
print(f'  {best_cfg_hp}')

final_mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[VARIANT]))
final_pnet = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub,
    G=auction.num_goods, hidden=cfg['policy_hidden'],
    residual_init_std=cfg['residual_init_std'],
)

final_logs = nested_GD_budget(
    auction=auction,
    mechanism_network=final_mnet, policy_network=final_pnet,
    profit_goal=cfg['profit_goal'],
    num_outer_epochs=cfg['num_outer_epochs'],
    lr_theta=best_cfg_hp['lr_theta'],
    num_inner_steps=best_cfg_hp['num_inner_steps'],
    lr_w=best_cfg_hp['lr_w'],
    num_samples_train=cfg['num_samples_train'],
    alpha_weight=best_cfg_hp['alpha_weight'],
    beta_weight=best_cfg_hp['beta_weight'],
    print_iter=cfg['print_iter'],
    seed=SEED,
)
final_mechanism_params = final_logs.pop('mechanism_params')
final_policy_params    = final_logs.pop('policy_params')
print('Final training complete.')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].plot(final_logs['reg_plus']); axes[0].set_title('Regret+');    axes[0].set_xlabel('Epoch')
axes[1].plot(final_logs['gap_plus']); axes[1].set_title('Profit Gap+'); axes[1].set_xlabel('Epoch')
axes[2].plot(final_logs['ir']);        axes[2].set_title('IR Loss');     axes[2].set_xlabel('Epoch')
axes[3].plot(final_logs['loss']);      axes[3].set_title('Total Loss');  axes[3].set_xlabel('Epoch')
plt.suptitle(f'Exp 2 — Variant {VARIANT} (optimal hparams)', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
final_eval_keys = jax.random.split(jax.random.PRNGKey(88888), 32)
final_results = eval_budget_mechanism_multikey(
    auction=auction, mechanism_network=final_mnet,
    mechanism_params=final_mechanism_params,
    profit_goal=cfg['profit_goal'],
    num_samples=256, keys=final_eval_keys,
    policy_network=final_pnet, policy_params=final_policy_params,
)
print_eval_report(final_results, prefix=f'Exp 2 Variant {VARIANT} (optimal hparams): ')


## Cross-Variant Comparison

Train variants A, B, C under the same optimal hyperparameters and compare all metrics. This is the core experimental comparison: does richer history compression help?

In [ ]:
variant_results = {
    VARIANT: {'logs': final_logs, 'eval': final_results,
              'mech_params': final_mechanism_params, 'pol_params': final_policy_params}
}

for _v in ['A', 'B', 'C']:
    if _v == VARIANT:
        continue
    print(f'\n--- Training Variant {_v} with optimal hparams ---')
    _vnet  = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[_v]))
    _vpnet = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=cfg['policy_hidden'],
        residual_init_std=cfg['residual_init_std'],
    )
    _vlogs = nested_GD_budget(
        auction=auction,
        mechanism_network=_vnet, policy_network=_vpnet,
        profit_goal=cfg['profit_goal'],
        num_outer_epochs=cfg['num_outer_epochs'],
        lr_theta=best_cfg_hp['lr_theta'],
        num_inner_steps=best_cfg_hp['num_inner_steps'],
        lr_w=best_cfg_hp['lr_w'],
        num_samples_train=cfg['num_samples_train'],
        alpha_weight=best_cfg_hp['alpha_weight'],
        beta_weight=best_cfg_hp['beta_weight'],
        print_iter=cfg['print_iter'],
        seed=SEED,
    )
    _vmp = _vlogs.pop('mechanism_params'); _vpp = _vlogs.pop('policy_params')
    _vek = jax.random.split(jax.random.PRNGKey(88888), 32)
    _vev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=_vnet, mechanism_params=_vmp,
        profit_goal=cfg['profit_goal'],
        num_samples=256, keys=_vek,
        policy_network=_vpnet, policy_params=_vpp,
    )
    print_eval_report(_vev, prefix=f'Variant {_v}: ')
    variant_results[_v] = {'logs': _vlogs, 'eval': _vev,
                            'mech_params': _vmp, 'pol_params': _vpp}

# Summary table
print('\n' + '='*72)
print(f'Exp 2 — Cross-Variant Comparison (optimal hparams)')
print('='*72)
print(f'  {"Variant":>7} | {"Revenue":>9} | {"Gap+":>7} | {"IR Loss":>9} | {"BV Rate":>8} | {"IV Rate":>8}')
print('  ' + '-'*65)
for _v in ['A', 'B', 'C']:
    if _v not in variant_results:
        continue
    _ev  = variant_results[_v]['eval']
    _rev = _ev['profit_stats']['mean']
    _gp  = _ev['gap_plus_stats']['mean']
    _ir  = _ev['ir_loss_stats']['mean']
    _bv  = _ev['budget_viol_stats']['mean']
    _iv  = _ev['ir_viol_stats']['mean']
    print(f'  {_v:>7} | {_rev:>9.4f} | {_gp:>7.4f} | {_ir:>9.4f} | {_bv:>8.4f} | {_iv:>8.4f}')

# Learning-curve comparison plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for _v, _col in zip(['A', 'B', 'C'], ['tab:blue', 'tab:orange', 'tab:green']):
    if _v not in variant_results:
        continue
    _l = variant_results[_v]['logs']
    axes[0].plot(_l['reg_plus'], color=_col, label=_v, alpha=0.85)
    axes[1].plot(_l['gap_plus'], color=_col, label=_v, alpha=0.85)
    axes[2].plot(_l['ir'],       color=_col, label=_v, alpha=0.85)
for _ax, _ttl in zip(axes, ['Regret+', 'Profit Gap+', 'IR Loss']):
    _ax.set_title(_ttl); _ax.set_xlabel('Epoch'); _ax.legend()
plt.suptitle(f'Exp 2 — A vs B vs C (optimal hparams)', y=1.02)
plt.tight_layout(); plt.show()


## Save All Outputs

In [ ]:
final_save_dir = f'outputs/private_budget_multigood/seed{SEED}/optimal'
os.makedirs(final_save_dir, exist_ok=True)

# Save model params, logs, and eval for each variant
variant_summary = {}
for _v, _vr in variant_results.items():
    with open(f'{final_save_dir}/mech_params_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['mech_params'], _f)
    with open(f'{final_save_dir}/pol_params_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['pol_params'], _f)
    with open(f'{final_save_dir}/logs_{_v}.pkl', 'wb') as _f:
        pickle.dump(_vr['logs'], _f)
    _ev = _vr['eval']
    variant_summary[_v] = {
        'revenue':        _ev['profit_stats'],
        'gap_plus':       _ev['gap_plus_stats'],
        'ir_loss':        _ev['ir_loss_stats'],
        'budget_viol':    _ev['budget_viol_stats'],
        'ir_viol':        _ev['ir_viol_stats'],
        'regret':         _ev.get('regret_stats', {}),
        'optimal_config': best_cfg_hp,
    }

with open(f'{final_save_dir}/variant_comparison.json', 'w') as _f:
    json.dump(variant_summary, _f, indent=2)

# Standalone optimal-hparams file for easy reference
with open(f'{final_save_dir}/optimal_hparams.json', 'w') as _f:
    json.dump({
        'exp':            'Exp 2',
        'variant':        VARIANT,
        'optimal_config': best_cfg_hp,
        'search_stats': {
            'num_trials':  len(all_trial_results),
            'best_score':  best_trial['score'],
            'best_revenue': best_trial['revenue_mean'],
        },
    }, _f, indent=2)

print(f'All outputs saved to: {final_save_dir}/')
print(f'  mech_params_{{A,B,C}}.pkl  — mechanism parameters per variant')
print(f'  pol_params_{{A,B,C}}.pkl   — policy parameters per variant')
print(f'  logs_{{A,B,C}}.pkl          — training loss history per variant')
print(f'  variant_comparison.json   — cross-variant evaluation summary')
print(f'  optimal_hparams.json      — best hyperparameter configuration')
print(f'HP search artefacts: {hp_save_dir}/')
print(f'  trial_results.json        — all trial records')
print(f'  best_hparams.json         — ranked configs + best config')
